<a id="sec-portada"></a>
# Guía de uso de Symbolic Operator Calculus<br>para el problema de Haseman con shift

## Wiener–Hopf relativo, kernels y primera reducción de Schur para $m=2$

**Enrique Díaz Ocampo**  
**Doctorado en Matemáticas**

Esta libreta es una guía práctica y reproducible para explorar el prototipo simbólico desarrollado alrededor del problema de contorno de Haseman con shift y de geometrías tipo *pure cusp stars*. Reúne el AST no conmutativo, la realización Fourier, las factorizaciones Wiener–Hopf relativas y la primera reducción de Schur disponible para $m=2$.

**Generación:** 16 de julio de 2026  
**Commit de referencia:** `3216465f6afd0fa97c627c73ba4fba95110dc179`

> **Advertencia de investigación.** Este software es un prototipo verificable dentro de hipótesis y modelos concretos. Sus cálculos simbólicos no sustituyen las demostraciones analíticas de existencia, acotación, invertibilidad o compacidad que exige la tesis.

In [1]:
print("Guía reproducible asociada al commit 3216465f6afd")
print("Ámbito: prototipo simbólico para m=2; no es una prueba analítica completa.")

Guía reproducible asociada al commit 3216465f6afd
Ámbito: prototipo simbólico para m=2; no es una prueba analítica completa.


<a id="sec-alcance"></a>
## 1. Qué resuelve actualmente la paquetería

El paquete representa palabras no conmutativas, aplica operadores desde la derecha, construye kernels Fourier normalizados, mantiene integrales formales y verifica las tres formas canónicas del operador Wiener–Hopf relativo. También produce la primera reducción de Schur para $m=2$ y proyecciones LaTeX trazables.

El alcance es deliberadamente limitado: no resuelve $m>2$, no invierte operadores, no decide compacidad y no convierte una expresión arbitraria del usuario en un operador válido. En particular, el regularizador $R_{1,1}$ y las funciones características permanecen formales.

In [2]:
capacidades = (
    ("AST no conmutativo", "Disponible"),
    ("Kernels Fourier normalizados", "Disponible"),
    ("Wiener–Hopf relativo y tres evidencias", "Disponible"),
    ("Primera reducción de Schur para m=2", "Disponible"),
    ("Inversión, compacidad y m>2", "No disponibles"),
)
for capacidad, estado in capacidades:
    print(f"{capacidad:45} {estado}")

AST no conmutativo                            Disponible
Kernels Fourier normalizados                  Disponible
Wiener–Hopf relativo y tres evidencias        Disponible
Primera reducción de Schur para m=2           Disponible
Inversión, compacidad y m>2                   No disponibles


<a id="sec-entorno"></a>
## 2. Preparación del entorno

La siguiente celda importa exclusivamente la fachada pública del paquete y registra versiones sin revelar rutas personales. Todos los ejemplos posteriores reutilizan los mismos símbolos con hipótesis explícitas.

In [3]:
from html import escape
from pathlib import Path
from tempfile import TemporaryDirectory
import subprocess
import sys

import sympy as sp
from IPython.display import HTML, Math, Markdown, display
import symbolic_operator_calculus as soc


def tabla(encabezados, filas):
    cabecera = "".join(f"<th>{escape(str(valor))}</th>" for valor in encabezados)
    cuerpo = "".join(
        "<tr>" + "".join(f"<td>{valor}</td>" for valor in fila) + "</tr>"
        for fila in filas
    )
    display(HTML(
        "<table style='border-collapse:collapse;width:100%;font-size:15px'>"
        f"<thead><tr>{cabecera}</tr></thead><tbody>{cuerpo}</tbody></table>"
    ))


def mostrar_formula(titulo, expresion):
    display(Markdown(f"**{titulo}**"))
    display(Math(sp.latex(expresion)))


repo_name = Path(soc.__file__).resolve().parents[2].name
commit = subprocess.check_output(
    ["git", "rev-parse", "--short=12", "HEAD"], text=True
).strip()
gamma1, gamma2, d = sp.symbols(
    "gamma_1 gamma_2 d", real=True, positive=True, finite=True
)
x, y, u, v = sp.symbols("x y u v", real=True, positive=True)
t, lam = sp.symbols("t lambda", real=True)
f = sp.Function("f")

tabla(
    ("Componente", "Valor"),
    (
        ("Proyecto", repo_name),
        ("Python", sys.version.split()[0]),
        ("SymPy", sp.__version__),
        ("Commit", commit),
        ("Importación", "symbolic_operator_calculus (API pública)"),
    ),
)

Componente,Valor
Proyecto,symbolic-operator-calculus
Python,3.10.14
SymPy,1.14.0
Commit,3216465f6afd
Importación,symbolic_operator_calculus (API pública)


Para iniciar la guía desde la raíz del repositorio:

```bash
PYENV_VERSION=3.10.14 PYTHONPATH=src jupyter lab
```

La ejecución automatizada de esta versión se realiza con `nbconvert` y un kernel limpio.

<a id="sec-convenciones"></a>
## 3. Convenciones matemáticas fundamentales

La composición se lee como

$$
(AB)f=A(Bf),
$$

de modo que el factor situado más a la derecha actúa primero. La transformada y su inversa son

$$
(\mathcal Ff)(\lambda)=\int_{\mathbb R}f(t)e^{-it\lambda}\,dt,
\qquad
(\mathcal F^{-1}g)(t)=\frac{1}{2\pi}\int_{\mathbb R}g(\lambda)e^{it\lambda}\,d\lambda.
$$

El AST conserva el orden y no presupone conmutatividad. `Product(())` es la identidad estructural, mientras `LinearCombination(())` es el cero estructural. Los coeficientes públicos son escalares concretos finitos. Además, $A=B$ denota igualdad exacta, mientras $A\simeq B$ registra una equivalencia módulo compactos: estas relaciones no son intercambiables.

In [4]:
assert soc.forward_exponential(t, lam) == sp.exp(-sp.I * t * lam)
assert soc.inverse_exponential(t, lam) == sp.exp(sp.I * t * lam)
assert soc.inverse_prefactor() == 1 / (2 * sp.pi)
assert soc.Product(()) != soc.Product((soc.I,))
assert soc.LinearCombination(()).terms == ()
print("Composición: el factor derecho actúa primero.")
print("Convención Fourier: exp(-i t lambda), inversa con 1/(2 pi).")
print("Identidad y cero estructurales: verificados como objetos distintos.")

Composición: el factor derecho actúa primero.
Convención Fourier: exp(-i t lambda), inversa con 1/(2 pi).
Identidad y cero estructurales: verificados como objetos distintos.


<a id="sec-ast"></a>
## 4. Primer contacto con el AST

El propósito es inspeccionar una palabra ordenada, un término y una combinación lineal sin convertirlos en productos conmutativos de SymPy. Usaremos átomos públicos cuyo renderer ya conoce.

In [5]:
producto = soc.Product((soc.G1, soc.Vtilde_alpha1))
producto_invertido = soc.Product((soc.Vtilde_alpha1, soc.G1))
termino = soc.Term(1, producto)
combinacion = soc.LinearCombination((
    termino,
    soc.Term(-1, producto_invertido),
))
identidad = soc.Product(())
cero = soc.LinearCombination(())
termino_identidad = soc.Term(1, soc.I)
reglas = soc.mvp_atomic_rules()

accion = soc.apply_product(producto, f(x), x, reglas)
accion_invertida = soc.apply_product(producto_invertido, f(x), x, reglas)
accion_identidad = soc.apply_product(identidad, f(x), x, reglas)
accion_cero = soc.apply_linear_combination(cero, f(x), x, reglas)

tabla(
    ("Objeto", "Estructura / LaTeX"),
    (
        ("Factores de Product", " → ".join(atomo.name for atomo in producto.factors)),
        ("Term", rf"\({soc.render_term_latex(termino)}\)"),
        ("LinearCombination", rf"\({soc.render_linear_combination_latex(combinacion)}\)"),
        ("Product(())", rf"\({soc.render_product_latex(identidad)}\)"),
        ("LinearCombination(())", rf"\({soc.render_linear_combination_latex(cero)}\)"),
        ("Term(1, I)", rf"\({soc.render_term_latex(termino_identidad)}\)"),
    ),
)
mostrar_formula("Acción de G1 · Ṽα1", accion)
mostrar_formula("Acción del producto invertido", accion_invertida)
assert accion != accion_invertida
assert accion_identidad == f(x)
assert accion_cero == 0
print("El orden cambia la acción; identidad y cero actúan correctamente.")

Objeto,Estructura / LaTeX
Factores de Product,G1 → Vtilde_alpha1
Term,"\(\left(G_{1}\,I\right)\,\widetilde V_{\alpha_1}\)"
LinearCombination,"\(\left(G_{1}\,I\right)\,\widetilde V_{\alpha_1} - \widetilde V_{\alpha_1}\,\left(G_{1}\,I\right)\)"
Product(()),\(I\)
LinearCombination(()),\(0\)
"Term(1, I)",\(I\)


**Acción de G1 · Ṽα1**

<IPython.core.display.Math object>

**Acción del producto invertido**

<IPython.core.display.Math object>

El orden cambia la acción; identidad y cero actúan correctamente.


<a id="sec-fourier"></a>
## 5. Kernels Fourier y Wiener–Hopf normalizados

Con $d>0$, la API evalúa las integrales inversas sobre las semirrectas correctas y obtiene

$$
K^+_{1,2}(t)=\frac{1}{2\pi(d-it)},
\qquad
K^-_{2,1}(t)=\frac{1}{2\pi(d+it)}.
$$

Los símbolos `bplus_symbol` y `bminus_symbol` son normalizados; el kernel original del artículo conserva los factores $2$ y $-2$.

In [6]:
k_plus = soc.kplus_kernel(t, d, lam)
k_minus = soc.kminus_kernel(t, d, lam)
h_12 = soc.article_wiener_hopf_kernel(t, -d)
h_21 = soc.article_wiener_hopf_kernel(t, d)
b_plus = soc.bplus_symbol(d, lam)
b_minus = soc.bminus_symbol(d, lam)

mostrar_formula("Símbolo normalizado positivo", b_plus)
mostrar_formula("Kernel K⁺₁,₂", k_plus)
mostrar_formula("Símbolo normalizado negativo", b_minus)
mostrar_formula("Kernel K⁻₂,₁", k_minus)

check_plus = sp.simplify(h_12 - 2 * k_plus)
check_minus = sp.simplify(h_21 + 2 * k_minus)
tabla(
    ("Correspondencia", "Resultado de simplify"),
    (("h₁,₂ − 2K⁺", check_plus), ("h₂,₁ + 2K⁻", check_minus)),
)
assert check_plus == check_minus == 0
print("Los factores originales 2 y -2 quedan preservados.")

**Símbolo normalizado positivo**

<IPython.core.display.Math object>

**Kernel K⁺₁,₂**

<IPython.core.display.Math object>

**Símbolo normalizado negativo**

<IPython.core.display.Math object>

**Kernel K⁻₂,₁**

<IPython.core.display.Math object>

Correspondencia,Resultado de simplify
"h₁,₂ − 2K⁺",0
"h₂,₁ + 2K⁻",0


Los factores originales 2 y -2 quedan preservados.


<a id="sec-factor12"></a>
## 6. Factorización Wiener–Hopf relativa $(1\to2)$

Partimos del operador exacto

$$
V_{\alpha_1}W(b_{1,2})V_{\alpha_2}^{-1}
=V_{\beta_{1,2}}W(b^{\mathrm L}_{1,2})
=W(b^{\mathrm R}_{1,2})V_{\beta_{1,2}},
\qquad
\beta_{1,2}(x)=\frac{\gamma_1}{\gamma_2}x.
$$

La tabla siguiente resume tipos y orden sin volcar las dataclasses completas.

In [7]:
trace12 = soc.build_relative_wiener_hopf_trace(
    1, 2, gamma1, gamma2, sp.Integer(0), d
)
model12 = trace12.factorization
identity12 = trace12.identity
rendered12 = soc.render_relative_wiener_hopf_trace_latex(trace12)
relative12_by_key = {step.key: step for step in rendered12.steps}


def etiqueta_factor(factor):
    if isinstance(factor, soc.DilationOperatorModel):
        mapa = factor.dilation
        direccion = "inversa" if factor.inverse else "directa"
        return f"{type(factor).__name__}: {mapa.kind}, {direccion}, escala={sp.sstr(mapa.scale)}"
    return f"{type(factor).__name__}: placement={factor.placement}"


filas = []
for nombre, producto_relativo in (
    ("Original", identity12.original),
    ("Izquierda", identity12.left),
    ("Derecha", identity12.right),
):
    filas.append((nombre, "<br>".join(etiqueta_factor(factor) for factor in producto_relativo.factors)))
tabla(("Producto", "Factores en orden escrito"), filas)

tabla(
    ("Dato", "Valor"),
    (
        ("Par", f"({model12.k}, {model12.j})"),
        ("Escalas", rf"\(\gamma_k={sp.latex(model12.gamma_k)},\;\gamma_j={sp.latex(model12.gamma_j)}\)"),
        ("Razón relativa", rf"\({sp.latex(model12.relative_scale)}\)"),
        ("identity.exact", identity12.exact),
    ),
)
display(Math(relative12_by_key["exact_identity"].latex))
assert identity12.exact is True

Producto,Factores en orden escrito
Original,"DilationOperatorModel: branch, directa, escala=gamma_1WienerHopfOperatorModel: placement=originalDilationOperatorModel: branch, inversa, escala=gamma_2"
Izquierda,"DilationOperatorModel: relative, directa, escala=gamma_1/gamma_2WienerHopfOperatorModel: placement=left"
Derecha,"WienerHopfOperatorModel: placement=rightDilationOperatorModel: relative, directa, escala=gamma_1/gamma_2"


Dato,Valor
Par,"(1, 2)"
Escalas,"\(\gamma_k=\gamma_{1},\;\gamma_j=\gamma_{2}\)"
Razón relativa,\(\frac{\gamma_{1}}{\gamma_{2}}\)
identity.exact,True


<IPython.core.display.Math object>

<a id="sec-evidencias"></a>
## 7. Las tres evidencias del operador relativo

Esta separación es central: ninguna bandera sustituye a las otras.

### 7.1 Evidencia estructural

`identity.exact` certifica que las tres formas tienen orden, índices, placements y procedencia L1 coherentes. No afirma por sí sola que se hayan aplicado a una función.

In [8]:
tabla(
    ("Evidencia", "Valor", "Significado"),
    (("identity.exact", identity12.exact, "Coherencia estructural de las tres formas"),),
)
assert identity12.exact is True

Evidencia,Valor,Significado
identity.exact,True,Coherencia estructural de las tres formas


<a id="sec-actions-evidence"></a>
### 7.2 Evidencia de acciones

`verify_relative_product_actions` aplica independientemente los tres productos y normaliza cada integral. Para la forma original, el cambio $u=\gamma_jy$ produce exactamente el jacobiano $\gamma_j$.

In [9]:
action_verification12 = soc.verify_relative_product_actions(
    identity12, f(y), output_variable=x, input_variable=y
)
acciones12 = (
    ("Original", action_verification12.original),
    ("Izquierda", action_verification12.left),
    ("Derecha", action_verification12.right),
)
tabla(
    ("Placement", "Kernel normalizado"),
    tuple((nombre, rf"\({sp.latex(accion.kernel)}\)") for nombre, accion in acciones12),
)
igualdades_kernel12 = tuple(
    sp.simplify(accion.kernel - action_verification12.original.kernel) == 0
    for _, accion in acciones12
)
print("Kernels iguales:", igualdades_kernel12)
print("actions_verified:", action_verification12.actions_verified)
assert all(igualdades_kernel12)
assert action_verification12.actions_verified is True

Placement,Kernel normalizado
Original,\(\frac{i \gamma_{2}}{\pi \left(i d + \gamma_{1} x - \gamma_{2} y\right)}\)
Izquierda,\(\frac{i \gamma_{2}}{\pi \left(i d + \gamma_{1} x - \gamma_{2} y\right)}\)
Derecha,\(\frac{i \gamma_{2}}{\pi \left(i d + \gamma_{1} x - \gamma_{2} y\right)}\)


Kernels iguales: (True, True, True)
actions_verified: True


<a id="sec-fourier-evidence"></a>
### 7.3 Evidencia Fourier

`verify_relative_symbol_correspondences` vuelve a calcular los símbolos desde `original_symbol` y los kernels desde `original_kernel`. Los campos almacenados por L1 son únicamente los resultados esperados contra los cuales se compara; no se reutilizan como cálculo.

In [10]:
symbol_verification12 = soc.verify_relative_symbol_correspondences(identity12)
tabla(
    ("Placement", "Calculado", "Almacenado", "Escala"),
    (
        (
            "Izquierda",
            rf"\({sp.latex(symbol_verification12.computed_left_symbol)}\)",
            rf"\({sp.latex(symbol_verification12.stored_left_symbol)}\)",
            rf"\({sp.latex(symbol_verification12.left_scale)}\)",
        ),
        (
            "Derecha",
            rf"\({sp.latex(symbol_verification12.computed_right_symbol)}\)",
            rf"\({sp.latex(symbol_verification12.stored_right_symbol)}\)",
            rf"\({sp.latex(symbol_verification12.right_scale)}\)",
        ),
    ),
)
mostrar_formula("Kernel izquierdo reconstruido", symbol_verification12.computed_left_kernel)
mostrar_formula("Kernel derecho reconstruido", symbol_verification12.computed_right_kernel)
print("Verificaciones parciales:",
      symbol_verification12.left_symbol_verified,
      symbol_verification12.right_symbol_verified,
      symbol_verification12.left_kernel_verified,
      symbol_verification12.right_kernel_verified)
print("correspondences_verified:", symbol_verification12.correspondences_verified)
assert symbol_verification12.correspondences_verified is True

Placement,Calculado,Almacenado,Escala
Izquierda,\(2 \chi_{plus}{\left(\lambda \right)} e^{- \frac{d \lambda}{\gamma_{2}}}\),\(2 \chi_{plus}{\left(\lambda \right)} e^{- \frac{d \lambda}{\gamma_{2}}}\),\(\gamma_{2}\)
Derecha,\(2 \chi_{plus}{\left(\lambda \right)} e^{- \frac{d \lambda}{\gamma_{1}}}\),\(2 \chi_{plus}{\left(\lambda \right)} e^{- \frac{d \lambda}{\gamma_{1}}}\),\(\gamma_{1}\)


**Kernel izquierdo reconstruido**

<IPython.core.display.Math object>

**Kernel derecho reconstruido**

<IPython.core.display.Math object>

Verificaciones parciales: True True True True
correspondences_verified: True


<a id="sec-actions"></a>
## 8. Aplicación independiente de los productos

Aplicamos las tres palabras a la función indefinida $f$. Cada acción directa conserva su propio cambio de variables; la forma normalizada permite comparar los kernels en una variable común. En el producto original, $u=\gamma_2 y$ explica el factor jacobiano $\gamma_2$.

In [11]:
for nombre, accion in acciones12:
    display(Markdown(f"**{nombre}: acción directa**"))
    display(Math(sp.latex(accion.direct_expression)))
    display(Markdown(f"**{nombre}: acción normalizada**"))
    display(Math(sp.latex(accion.normalized_expression)))

kernel_comun12 = action_verification12.original.kernel
assert all(sp.simplify(accion.kernel - kernel_comun12) == 0 for _, accion in acciones12)
print("Los tres kernels extraídos son algebraicamente iguales.")

**Original: acción directa**

<IPython.core.display.Math object>

**Original: acción normalizada**

<IPython.core.display.Math object>

**Izquierda: acción directa**

...

**Derecha: acción directa**

<IPython.core.display.Math object>

**Derecha: acción normalizada**

<IPython.core.display.Math object>

Los tres kernels extraídos son algebraicamente iguales.


<a id="sec-factor21"></a>
## 9. Dirección $(2\to1)$

La dirección inversa conserva el signo negativo, el factor $-2$, el soporte $\chi_-$ y la razón relativa $\gamma_2/\gamma_1$. Repetimos sólo las comprobaciones esenciales.

In [12]:
trace21 = soc.build_relative_wiener_hopf_trace(
    2, 1, gamma2, gamma1, d, sp.Integer(0)
)
model21 = trace21.factorization
action_verification21 = soc.verify_relative_product_actions(trace21.identity)
symbol_verification21 = soc.verify_relative_symbol_correspondences(trace21.identity)
frequency21 = model21.frequency_variable

mostrar_formula("Símbolo original 2→1", model21.original_symbol)
mostrar_formula("Símbolo izquierdo 2→1", symbol_verification21.computed_left_symbol)
mostrar_formula("Símbolo derecho 2→1", symbol_verification21.computed_right_symbol)
tabla(
    ("Invariante", "Comprobación"),
    (
        ("Factor", symbol_verification21.computed_left_symbol.as_coeff_Mul()[0]),
        ("Soporte χ₋", symbol_verification21.computed_left_symbol.has(soc.chi_minus(frequency21))),
        ("Razón relativa", rf"\({sp.latex(model21.relative_scale)}\)"),
        ("identity.exact", trace21.identity.exact),
        ("actions_verified", action_verification21.actions_verified),
        ("correspondences_verified", symbol_verification21.correspondences_verified),
    ),
)
assert symbol_verification21.computed_left_symbol.as_coeff_Mul()[0] == -2
assert symbol_verification21.computed_left_symbol.has(soc.chi_minus(frequency21))
assert trace21.identity.exact and action_verification21.actions_verified
assert symbol_verification21.correspondences_verified

**Símbolo original 2→1**

<IPython.core.display.Math object>

**Símbolo izquierdo 2→1**

<IPython.core.display.Math object>

**Símbolo derecho 2→1**

<IPython.core.display.Math object>

Invariante,Comprobación
Factor,-2
Soporte χ₋,True
Razón relativa,\(\frac{\gamma_{2}}{\gamma_{1}}\)
identity.exact,True
actions_verified,True
correspondences_verified,True


<a id="sec-bloques"></a>
## 10. Bloques $A_{1,2}$ y $A_{2,1}$

Los objetos de bloque distinguen el bloque exacto, su modelo de Wiener–Hopf normalizado y la relación módulo compactos. Estas relaciones no son las identidades exactas del dominio relativo.

> **En el estado actual, el dominio Wiener–Hopf relativo y el AST de Schur son sistemas validados pero todavía separados.**

In [13]:
relation12 = soc.a12_mod_compact_relation()
relation21 = soc.a21_mod_compact_relation()
model_block12 = soc.a12_wh_model()
model_block21 = soc.a21_wh_model()

tabla(
    ("Bloque", "Relación", "Coeficientes del modelo", "Tipo"),
    (
        ("A₁,₂", "módulo compactos (≃)", tuple(term.coefficient for term in model_block12.expression.terms), type(relation12).__name__),
        ("A₂,₁", "módulo compactos (≃)", tuple(term.coefficient for term in model_block21.expression.terms), type(relation21).__name__),
    ),
)
display(Math(r"A_{1,2}\simeq " + soc.render_linear_combination_latex(model_block12.expression)))
display(Math(r"A_{2,1}\simeq " + soc.render_linear_combination_latex(model_block21.expression)))
assert relation12.exact != relation12.model
assert relation21.exact != relation21.model
print("Los factores 2/-2 del símbolo original no son coeficientes de estos bloques normalizados.")

Bloque,Relación,Coeficientes del modelo,Tipo
"A₁,₂",módulo compactos (≃),"(1, -1)",ModCompactRelation
"A₂,₁",módulo compactos (≃),"(-1, 1)",ModCompactRelation


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Los factores 2/-2 del símbolo original no son coeficientes de estos bloques normalizados.


<a id="sec-schur"></a>
## 11. Primera reducción de Schur para $m=2$

La relación exacta almacenada es

$$
A_{22}^{(1)}=A_{22}-A_{21}R_{11}A_{12}.
$$

Después se sustituyen los modelos fuera de la diagonal **módulo compactos**. Como el modelo de $A_{21}$ ya lleva un signo negativo, el signo exterior de Schur produce una corrección positiva y la expansión conserva el patrón $(+,-,-,+)$.

> $R_{11}$ es un regularizador formal; el paquete no lo trata como inverso exacto ni aplica cancelaciones.

In [14]:
schur_trace = soc.build_first_schur_derivation_trace(
    f(x), x,
    input_variable=y,
    outer_variable=u,
    middle_variable=v,
    rules=soc.mvp_atomic_rules(),
)
rendered_schur = soc.render_first_schur_derivation_latex(schur_trace)
schur_by_key = {step.key: step for step in rendered_schur.steps}

for clave in ("exact_definition", "offdiagonal_models", "correction_sign", "correction_expansion"):
    display(Markdown(f"**{schur_by_key[clave].title}**"))
    display(Math(schur_by_key[clave].latex))

tabla(
    ("Dato", "Valor"),
    (
        ("Signo exterior de Schur", schur_trace.sign_trace.schur_sign),
        ("Signo inicial de A₂,₁", schur_trace.sign_trace.leading_a21_sign),
        ("Signo de la corrección", schur_trace.sign_trace.correction_sign),
        ("Coeficientes expandidos", tuple(term.coefficient for term in schur_trace.correction.terms)),
        ("Regularizador", "R11 formal"),
    ),
)
assert tuple(term.coefficient for term in schur_trace.correction.terms) == (1, -1, -1, 1)

**Exact definition**

<IPython.core.display.Math object>

**Off-diagonal models**

<IPython.core.display.Math object>

**Correction sign**

<IPython.core.display.Math object>

**Correction expansion**

<IPython.core.display.Math object>

Dato,Valor
Signo exterior de Schur,-1
"Signo inicial de A₂,₁",-1
Signo de la corrección,1
Coeficientes expandidos,"(1, -1, -1, 1)"
Regularizador,R11 formal


<a id="sec-c22"></a>
## 12. Kernels $M_{2,1}$, $M_{1,2}$ y $C_{2,2}^{(1)}$

La factorización lateral conserva el orden de los kernels y genera variables mudas distintas. El kernel combinado es formal:

$$
C_{2,2}^{(1)}(x,y)=\int_0^\infty\!\int_0^\infty
M_{2,1}(x,u)R_{1,1}(u,v)M_{1,2}(v,y)\,dv\,du.
$$

In [15]:
for clave in ("left_kernel", "right_kernel", "combined_kernel"):
    display(Markdown(f"**{schur_by_key[clave].title}**"))
    display(Math(schur_by_key[clave].latex))

tabla(
    ("Objeto", "Variables"),
    (
        ("M₂,₁", f"libres: {sorted(str(s) for s in schur_trace.m21.free_symbols)}"),
        ("M₁,₂", f"libres: {sorted(str(s) for s in schur_trace.m12.free_symbols)}"),
        ("C₂,₂⁽¹⁾", f"libres: {sorted(str(s) for s in schur_trace.combined_kernel.free_symbols)}"),
        ("Integración", "v interior, u exterior; intervalo (0, ∞)"),
    ),
)
assert schur_trace.combined_kernel.has(sp.Function("R11"))
print("R11(u,v) permanece formal y el orden M21 · R11 · M12 se conserva.")

**Left kernel**

<IPython.core.display.Math object>

**Right kernel**

<IPython.core.display.Math object>

**Combined kernel**

<IPython.core.display.Math object>

Objeto,Variables
"M₂,₁","libres: ['gamma2', 'u', 'x']"
"M₁,₂","libres: ['gamma1', 'v', 'y']"
"C₂,₂⁽¹⁾","libres: ['gamma1', 'gamma2', 'x', 'y']"
Integración,"v interior, u exterior; intervalo (0, ∞)"


R11(u,v) permanece formal y el orden M21 · R11 · M12 se conserva.


<a id="sec-accion-schur"></a>
## 13. Acción del primer bloque reducido

La acción disponible mantiene separadas la parte diagonal y la corrección compacta. Puede inspeccionarse estructuralmente, proyectarse como expresión escalar y renderizarse, pero sus integrales formales no se evalúan numéricamente.

In [16]:
compact_action = schur_trace.compact_action
tabla(
    ("Vista", "Objeto"),
    (
        ("Estructural", type(compact_action).__name__),
        ("Relación", type(compact_action.relation).__name__ + " (módulo compactos)"),
        ("Diagonal", type(compact_action.diagonal).__name__),
        ("Corrección integral", type(compact_action.correction).__name__),
    ),
)
mostrar_formula("Parte diagonal aplicada", compact_action.diagonal.as_expr())
mostrar_formula("Corrección integral aplicada", compact_action.correction)
display(Markdown("**Vista LaTeX compacta de $(A_{22}^{(1)}f)(x)$**"))
display(Math(schur_by_key["compact_model_action"].latex))
assert compact_action.as_expr() == compact_action.diagonal.as_expr() + compact_action.correction

Vista,Objeto
Estructural,FirstSchurCompactModelAction
Relación,ModCompactSchurRelation (módulo compactos)
Diagonal,AppliedLinearCombination
Corrección integral,Integral


**Parte diagonal aplicada**

<IPython.core.display.Math object>

**Corrección integral aplicada**

<IPython.core.display.Math object>

**Vista LaTeX compacta de $(A_{22}^{(1)}f)(x)$**

<IPython.core.display.Math object>

<a id="sec-latex"></a>
## 14. Renderizado y exportación LaTeX

Los renderers proyectan objetos ya validados: no recalculan la matemática. El exportador recibe una derivación renderizada y escribe un documento en un directorio temporal.

> **LaTeX es una proyección; no es la identidad matemática del AST.**

In [17]:
latex_producto = soc.render_product_latex(producto)
latex_termino = soc.render_term_latex(soc.Term(-1, producto))
latex_relativo = rendered12.as_latex()
latex_schur = rendered_schur.as_latex()

tabla(
    ("Renderer", "Resultado resumido"),
    (
        ("Producto", rf"\({latex_producto}\)"),
        ("Término", rf"\({latex_termino}\)"),
        ("Traza relativa", f"{len(rendered12.steps)} pasos"),
        ("Derivación Schur", f"{len(rendered_schur.steps)} pasos"),
    ),
)
display(Math(relative12_by_key["exact_identity"].latex))
display(Math(schur_by_key["compact_model_action"].latex))

with TemporaryDirectory() as temporary_directory:
    destination = Path(temporary_directory) / "first-schur.tex"
    soc.export_first_schur_derivation_tex(rendered_schur, destination)
    exported_document = destination.read_text(encoding="utf-8")
    export_summary = (destination.name, len(exported_document), r"\begin{document}" in exported_document)

print("Exportación temporal:", export_summary)
assert export_summary[2] is True

Renderer,Resultado resumido
Producto,"\(\left(G_{1}\,I\right)\,\widetilde V_{\alpha_1}\)"
Término,"\(- \left(G_{1}\,I\right)\,\widetilde V_{\alpha_1}\)"
Traza relativa,10 pasos
Derivación Schur,9 pasos


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Exportación temporal: ('first-schur.tex', 3158, True)


<a id="sec-estudio"></a>
## 15. Qué estudiar después con esta herramienta

La guía permite formular preguntas de investigación controladas:

1. ¿Cómo cambia cada símbolo al variar $\gamma_k/\gamma_j$?
2. ¿Qué papel tiene el signo del salto $c_k-c_j$ en los soportes $\chi_\pm$?
3. ¿Cómo emerge el jacobiano al comparar las acciones original e izquierda?
4. ¿Qué descriptor común permitiría insertar los operadores relativos en $A_{1,2}$ y $A_{2,1}$ sin perder índices?
5. ¿Qué invariantes de dimensión, orden y procedencia hacen falta antes de abordar $m>2$?

Son preguntas para trabajo posterior, no resultados nuevos afirmados por el paquete.

In [18]:
parametros_estudio = {
    "razon_1_2": model12.relative_scale,
    "razon_2_1": model21.relative_scale,
    "jacobiano_1_2": model12.gamma_j,
    "soporte_1_2": "chi_plus",
    "soporte_2_1": "chi_minus",
}
for nombre, valor in parametros_estudio.items():
    print(f"{nombre}: {valor}")

razon_1_2: gamma_1/gamma_2
razon_2_1: gamma_2/gamma_1
jacobiano_1_2: gamma_2
soporte_1_2: chi_plus
soporte_2_1: chi_minus


<a id="sec-limitaciones"></a>
## 16. Limitaciones actuales

| Capacidad | Estado |
| --- | --- |
| AST no conmutativo | Disponible |
| Fourier normalizado | Disponible |
| Wiener–Hopf relativo | Disponible |
| Acciones relativas independientes | Disponible |
| Correspondencias Fourier verificadas | Disponible |
| Primer Schur ($m=2$) | Disponible |
| Integración relativa con Schur | Pendiente |
| Inversión real de operadores | No disponible |
| $m>2$ | No disponible |
| Evaluación numérica general | No disponible |

El paquete tampoco prueba por sí mismo la compacidad de restos, la existencia del regularizador ni las propiedades funcionales globales del problema de contorno.

In [19]:
limitaciones_criticas = (
    "Integración automática del dominio relativo con Schur: pendiente",
    "R11: regularizador formal, no inverso computado",
    "Compacidad: registrada como relación, no decidida por el software",
    "Generalización m>2: no disponible",
)
for item in limitaciones_criticas:
    print(f"- {item}")

- Integración automática del dominio relativo con Schur: pendiente
- R11: regularizador formal, no inverso computado
- Compacidad: registrada como relación, no decidida por el software
- Generalización m>2: no disponible


<a id="sec-resumen"></a>
## 17. Resumen reproducible

La celda final reúne comprobaciones públicas breves. No ejecuta la suite del repositorio ni intenta evaluar integrales formales.

In [20]:
assert identity12.exact is True
assert action_verification12.actions_verified is True
assert symbol_verification12.correspondences_verified is True
assert trace21.identity.exact is True
assert action_verification21.actions_verified is True
assert symbol_verification21.correspondences_verified is True
assert sp.simplify(h_12 - 2 * k_plus) == 0
assert sp.simplify(h_21 + 2 * k_minus) == 0
assert tuple(term.coefficient for term in schur_trace.correction.terms) == (1, -1, -1, 1)
assert schur_trace.combined_kernel.has(sp.Function("R11"))
print("Todas las comprobaciones del tutorial fueron satisfactorias.")

Todas las comprobaciones del tutorial fueron satisfactorias.


<a id="sec-galeria"></a>
## Galería visual de la guía

### Portada y alcance
Vista de las secciones 0–1, donde se fija el propósito doctoral y la frontera del prototipo.

![Portada y alcance](../docs/thesis_usage_guide/screenshots/01_portada_y_alcance.png)

### AST no conmutativo
La sección 4 muestra el orden estructural, su acción y los objetos identidad/cero.

![AST no conmutativo](../docs/thesis_usage_guide/screenshots/02_ast_no_conmutativo.png)

### Kernels Fourier
La sección 5 verifica los kernels normalizados y los factores originales $2/-2$.

![Kernels Fourier](../docs/thesis_usage_guide/screenshots/03_kernels_fourier.png)

### Factorización relativa $1\to2$
Resumen visual de los tres productos canónicos y de la razón $\gamma_1/\gamma_2$.

![Factorización relativa 1 a 2](../docs/thesis_usage_guide/screenshots/04_factorizacion_relativa_1_2.png)

### Acciones relativas
Las secciones 7.2 y 8 aplican y normalizan independientemente los tres productos.

![Acciones relativas](../docs/thesis_usage_guide/screenshots/05_acciones_relativas.png)

### Correspondencias Fourier
La sección 7.3 contrapone resultados calculados y almacenados para ambos placements.

![Correspondencias Fourier](../docs/thesis_usage_guide/screenshots/06_correspondencias_fourier.png)

### Factorización relativa $2\to1$
La sección 9 conserva signo negativo, factor $-2$, soporte $\chi_-$ y las tres evidencias.

![Factorización relativa 2 a 1](../docs/thesis_usage_guide/screenshots/07_factorizacion_relativa_2_1.png)

### Reducción de Schur $m=2$
La sección 11 distingue definición exacta, modelos módulo compactos y expansión de signos.

![Reducción Schur m2](../docs/thesis_usage_guide/screenshots/08_reduccion_schur_m2.png)

### Kernel combinado $C_{2,2}^{(1)}$
La sección 12 muestra $M_{2,1}$, $M_{1,2}$, el regularizador formal y el orden de integración.

![Kernel combinado C22](../docs/thesis_usage_guide/screenshots/09_kernel_combinado_c22.png)

### Renderizado LaTeX
La sección 14 ilustra renderers semánticos y una exportación temporal real.

![Renderizado LaTeX](../docs/thesis_usage_guide/screenshots/10_renderizado_latex.png)